In [20]:
import yaml
import tensorflow as tf
from pathlib import Path 
intent_file=Path("../../data/intent-classifier.yml")
INTENTS={}
with open(intent_file, 'r', encoding='utf-8') as file:
    INTENTS=yaml.safe_load(file)

In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np 

model =SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
def normalize(v):
    return v/np.linalg.norm(v)

In [23]:
all_texts = []
all_labels = []

for intent, examples in INTENTS.items():
    for ex in examples:
        all_texts.append(ex)
        all_labels.append(intent)

embeddings = model.encode(all_texts)
embeddings = np.array([normalize(v) for v in embeddings])


In [24]:

def predict_intent(text, threshold=0.55, margin_threshold=0.08):

    text_vec = model.encode(text)
    text_vec = normalize(text_vec)

    # similarity with all examples
    scores = np.dot(embeddings, text_vec)

    best_idx = np.argmax(scores)
    best_score = scores[best_idx]

    # second best for margin check
    sorted_scores = np.sort(scores)[::-1]
    margin = sorted_scores[0] - sorted_scores[1]

    best_intent = all_labels[best_idx]

    # fallback logic
    if best_score < threshold or margin < margin_threshold:
        return {
            "intent": "unknown",
            "confidence": float(best_score),
            "margin": float(margin)
        }

    return {
        "intent": best_intent,
        "confidence": float(best_score),
        "margin": float(margin)
    }



In [25]:
extracted_intent=predict_intent("how does this is works for me")
print(extracted_intent)

{'intent': 'information_request', 'confidence': 0.7770821452140808, 'margin': 0.2885296046733856}


LabelEncoder

tf.karas.layers.TextVectorization

tf.keras.Sequential

## Start Tensorflow


### Label Encoder example 

In [26]:
labels =['order_creation','cancel_order','order_status']
from sklearn.preprocessing import LabelEncoder
label_encoder =LabelEncoder()
encoded = label_encoder.fit_transform(labels)
print(label_encoder.classes_)
print(encoded)

['cancel_order' 'order_creation' 'order_status']
[1 0 2]


In [27]:
print(f'export value for index 1 {label_encoder.inverse_transform([1])}')
print(f'export value for index 1 {label_encoder.inverse_transform([0])}')



export value for index 1 ['order_creation']
export value for index 1 ['cancel_order']


### TextVectorization Example from tensorflow 

In [28]:
texts =[]
labels =[]

for label in INTENTS:
   for text in INTENTS[label]:
      texts.append(text)
      labels.append(label)


In [29]:
vectorizer =tf.keras.layers.TextVectorization(
    max_tokens=1000,
    output_mode="int",
    output_sequence_length=8
)

In [30]:
vectorizer.adapt(texts)

In [31]:
print(vectorizer.get_vocabulary())

['', '[UNK]', np.str_('order'), np.str_('this'), np.str_('i'), np.str_('to'), np.str_('a'), np.str_('new'), np.str_('want'), np.str_('my'), np.str_('me'), np.str_('for'), np.str_('you'), np.str_('can'), np.str_('the'), np.str_('item'), np.str_('dont'), np.str_('create'), np.str_('please'), np.str_('is'), np.str_('cancel'), np.str_('with'), np.str_('what'), np.str_('singara'), np.str_('product'), np.str_('make'), np.str_('about'), np.str_('purchase'), np.str_('place'), np.str_('more'), np.str_('it'), np.str_('how'), np.str_('generate'), np.str_('available'), np.str_('an'), np.str_('stop'), np.str_('process'), np.str_('one'), np.str_('need'), np.str_('move'), np.str_('information'), np.str_('delivery'), np.str_('buy'), np.str_('add'), np.str_('account'), np.str_('your'), np.str_('would'), np.str_('tell'), np.str_('take'), np.str_('service'), np.str_('report'), np.str_('project'), np.str_('payment'), np.str_('mind'), np.str_('like'), np.str_('if'), np.str_('give'), np.str_('forward'), np.

In [46]:
model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(input_dim=1000, output_dim=32),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(len(label_encoder.classes_), activation="softmax")
])


In [47]:
label_encoder =LabelEncoder()
encoded = label_encoder.fit_transform(labels)
print(label_encoder.classes_)
print(encoded)

['cancel_order' 'creation_request' 'information_request' 'order_creation']
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3]


In [48]:

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
x_train =tf.constant(texts,dtype=tf.string)
y_train =np.array(encoded, dtype=np.int32)
print(x_train.shape)
print(y_train.shape)


print("_--------------------------------------------------------")
print("Classes :", label_encoder.classes_)
print("Number of classes :", len(label_encoder.classes_))
print("Unique encoded labels :", np.unique(encoded))
print("Max Label  :", np.max(encoded))
model.fit(
    x_train,
    y_train,
    epochs=100,
    verbose=1
)


(94,)
(94,)
_--------------------------------------------------------
Classes : ['cancel_order' 'creation_request' 'information_request' 'order_creation']
Number of classes : 4
Unique encoded labels : [0 1 2 3]
Max Label  : 3
Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3404 - loss: 1.3850 
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6596 - loss: 1.3771
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7553 - loss: 1.3705
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7766 - loss: 1.3632 
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7766 - loss: 1.3553 
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7553 - loss: 1.3463 
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7979 - loss: 1.3364
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8085 - loss: 1.3254
Epoch 9/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8298 - loss: 1.3135
Epoch 10/100
3/3 

In [49]:
def predict_intent(text:str):
    prediction =model.predict(
        tf.constant([text], dtype=tf.string)
        ,verbose=0)[0]
    
    index =np.argmax(prediction)
 
    confidence = prediction[index]
    intent =label_encoder.inverse_transform([index])[0]


    return {
        'intent':intent,
        'confidence':confidence   
    }

In [50]:
predict_intent("I will not definitly go with old order")

{'intent': np.str_('cancel_order'), 'confidence': np.float32(0.83865607)}